In [2]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt



In [ ]:
# Pull Data from Github report for NFL seasons 2019-2025 - this is full game play by play data (takes a few minutes to load)
full_pbp_data_url = ("https://github.com/nflverse/nflverse-data/"
    "releases/download/pbp")

df = pd.concat(
    [
        pd.read_csv(
            f"{full_pbp_data_url}/play_by_play_{year}.csv",
            low_memory=False
        )
        for year in range (2015,2026)
    ],
    ignore_index=True
)


In [7]:
print (df.shape)
print(df.head())
print(df.columns.tolist())



(342249, 372)
   play_id          game_id  old_game_id home_team away_team season_type  \
0        1  2019_01_ATL_MIN   2019090804       MIN       ATL         REG   
1       36  2019_01_ATL_MIN   2019090804       MIN       ATL         REG   
2       51  2019_01_ATL_MIN   2019090804       MIN       ATL         REG   
3       79  2019_01_ATL_MIN   2019090804       MIN       ATL         REG   
4      100  2019_01_ATL_MIN   2019090804       MIN       ATL         REG   

   week posteam posteam_type defteam  ... out_of_bounds  home_opening_kickoff  \
0     1     NaN          NaN     NaN  ...             0                     0   
1     1     ATL         away     MIN  ...             0                     0   
2     1     ATL         away     MIN  ...             0                     0   
3     1     ATL         away     MIN  ...             0                     0   
4     1     ATL         away     MIN  ...             0                     0   

     qb_epa  xyac_epa  xyac_mean_yardage  

In [5]:
print(df.isnull().sum())

play_id                    0
game_id                    0
old_game_id                0
home_team                  0
away_team                  0
                       ...  
xyac_median_yardage    32447
xyac_success           32447
xyac_fd                32447
xpass                  11707
pass_oe                12752
Length: 372, dtype: int64


In [ ]:
#Separate out the first three quarters of each game for defensive stats
first_three_quarters = df[df["qtr"].isin([1,2,3])].copy()

#Build fumble data frame to track fumbles forced and fumbles lost
fumbles_pbp = first_three_quarters
multiple_fumbles = (
    fumbles_pbp["fumbled_2_team"].notna()
    | fumbles_pbp["fumbled_2_player_id"].notna()
)

single_fumble_losses = fumbles_pbp[
    fumbles_pbp["fumble_lost"].eq(1)
    & fumbles_pbp["fumbled_1_team"].notna()
    & ~multiple_fumbles
]

fumbles_lost_stats = (
    single_fumble_losses
    .groupby(["season", "game_id", "fumbled_1_team", "qtr"])
    .size()
    .reset_index(name="fumbles_lost_single_fumble_plays")
    .rename(columns={"fumbled_1_team": "team"})
)

#Build Return Touchdown data frame to track return touchdowns allowed
return_tds_pbp = first_three_quarters[
    first_three_quarters["return_touchdown"].eq(1)
    & first_three_quarters["td_team"].notna()
].copy()

return_td_stats = (
    return_tds_pbp
    .groupby(["season", "game_id", "td_team", "qtr"])
    .size()
    .reset_index(name="return_touchdowns_scored")
    .rename(columns={"td_team": "team"})
)

#separate out defensive stats for the first 3 quarters (group by Defensive team, season, game_id, and quarter)
defense_stats = (
    first_three_quarters
    .groupby(["season", "game_id", "defteam", "qtr"])
    .agg(
        sacks_made = ("sack", "sum"),
        interceptions_made = ("interception", "sum"),
        passing_yards_allowed = ("passing_yards", "sum"),
        rushing_yards_allows = ("rushing_yards", "sum"),
        #fumbles_forced = ("fumble_forced", "sum"),
        #fumbles_recovered = ("fumble_lost", "sum"),
        first_downs_rush_allowed = ("first_down_rush", "sum"),
        first_downs_pass_allowed = ("first_down_pass", "sum"),
        first_downs_penalty_allowed = ("first_down_penalty", "sum"),
        pass_touchdown_allowed = ("pass_touchdown", "sum"),
        rush_touchdown_allowed = ("rush_touchdown", "sum"),
        #return_touchdown_allowed = ("return_touchdown", "sum"),
        third_down_conversions_allowed=("third_down_converted", "sum"),
        third_down_stops=("third_down_failed", "sum"),
    )
    .reset_index()
    .rename(columns={"defteam": "team"})
)


#separate offensive states based on (possesion team, season, game_id, and quarter)
offense_stats = (
    first_three_quarters
    .groupby(["season", "game_id", "posteam", "qtr"])
        .agg(
            sacks_allowed = ("sack", "sum"),
            interceptions_thrown = ("interception", "sum"),
            passing_yards_gained = ("passing_yards", "sum"),
            rushing_yards_gained = ("rushing_yards", "sum"),
            #fumbles_lost = ("fumble_lost", "sum"),
            #offensive_fumbles_recovered = ("fumble_forced", "sum"),
            first_downs_rush_gained = ("first_down_rush", "sum"),
            first_downs_pass_gained = ("first_down_pass", "sum"),
            first_downs_penalty_gained = ("first_down_penalty", "sum"),
            pass_touchdown_gained = ("pass_touchdown", "sum"),
            rush_touchdown_gained = ("rush_touchdown", "sum"),
            third_down_conversions=("third_down_converted", "sum"),
            third_down_failures=("third_down_failed", "sum"),
        )
        .reset_index()
        .rename(columns={"posteam": "team"})
)

#Scoring Progression through 3 Quarters
# Put plays in chronological order within each game
ordered = first_three_quarters.sort_values(
    ["season", "game_id", "play_id"]
).copy()

score_cols = ["total_home_score", "total_away_score"]

# Carry forward the running score if an end-of-quarter row is missing it
ordered[score_cols] = (
    ordered.groupby(["season", "game_id"])[score_cols].ffill()
)

# Keep the last row of each quarter, including quarter-ending events
quarter_scores = (
    ordered
    .groupby(["season", "game_id", "qtr"])
    .tail(1)
    [[
        "season", "game_id", "qtr",
        "home_team", "away_team",
        "total_home_score", "total_away_score",
    ]]
    .rename(columns={
        "total_home_score": "home_score_quarter_end",
        "total_away_score": "away_score_quarter_end",
    })
    .reset_index(drop=True)
)

quarter_scores["home_points_scored_in_quarter"] = (
    quarter_scores.groupby(["season", "game_id"])["home_score_quarter_end"]
    .diff()
    .fillna(quarter_scores["home_score_quarter_end"])
)

quarter_scores["away_points_scored_in_quarter"] = (
    quarter_scores.groupby(["season", "game_id"])["away_score_quarter_end"]
    .diff()
    .fillna(quarter_scores["away_score_quarter_end"])
)

quarter_scores["home_team_point_diff"] = (
    quarter_scores["home_score_quarter_end"] - quarter_scores["away_score_quarter_end"]
)

quarter_scores["away_team_point_diff"] = (
    quarter_scores["away_score_quarter_end"] - quarter_scores["home_score_quarter_end"]
)

quarter_scores["home_team_qtr_point_change"] = (
    quarter_scores["home_points_scored_in_quarter"] - quarter_scores["away_points_scored_in_quarter"]
)

quarter_scores["away_team_qtr_point_change"] = (
    quarter_scores["away_points_scored_in_quarter"] - quarter_scores["home_points_scored_in_quarter"]
)

quarter_scores["combined_score_quarter_end"] = (
    quarter_scores["home_score_quarter_end"] + quarter_scores["away_score_quarter_end"]
)

quarter_scores["absolute_score_margin"] = (
    quarter_scores["home_team_point_diff"].abs()
)


#Build Red Zone Stats
#DF for all RedZone Plays
red_zone_plays = first_three_quarters[
    first_three_quarters["yardline_100"].between(1, 20)
    & first_three_quarters["down"].notna()
    & ~first_three_quarters["extra_point_attempt"].eq(1)
    & ~first_three_quarters["two_point_attempt"].eq(1)
].copy()

#Unique RedZone Visits
red_zone_trips = (
    red_zone_plays
    .sort_values(["season", "game_id", "play_id"])
    .drop_duplicates(
        subset=["season", "game_id", "posteam", "fixed_drive"]
    )
)

#Offensive RedZone Stats for total trips 
red_zone_offense_stats = (
    red_zone_trips
    .groupby(["season", "game_id", "posteam", "qtr"])
    .size()
    .reset_index(name="red_zone_trips")
    .rename(columns={"posteam": "team"})
)

#Defensive RedZone Stats for total trips allowed by defense
red_zone_defense_stats = (
    red_zone_trips
    .groupby(["season", "game_id", "defteam", "qtr"])
    .size()
    .reset_index(name="red_zone_trips_allowed")
    .rename(columns={"defteam": "team"})
)

#Offensive plays in the red zone resulting in a tochdown
red_zone_td_stats = (
    red_zone_plays
    .groupby(["season", "game_id", "posteam", "qtr"])
    .agg(
        red_zone_passing_tds=("pass_touchdown", "sum"),
        red_zone_rushing_tds=("rush_touchdown", "sum"),
    )
    .reset_index()
    .rename(columns={"posteam": "team"})
)

red_zone_td_stats["red_zone_touchdowns"] = (
    red_zone_td_stats["red_zone_passing_tds"]
    + red_zone_td_stats["red_zone_rushing_tds"]
)

#Defensive redzone stats for touchdowns allowed
red_zone_defense_td_stats = (
    red_zone_plays
    .groupby(["season", "game_id", "defteam", "qtr"])
    .agg(
        red_zone_passing_tds_allowed=("pass_touchdown", "sum"),
        red_zone_rushing_tds_allowed=("rush_touchdown", "sum"),
    )
    .reset_index()
    .rename(columns={"defteam": "team"})
)

red_zone_defense_td_stats["red_zone_touchdowns_allowed"] = (
    red_zone_defense_td_stats["red_zone_passing_tds_allowed"]
    + red_zone_defense_td_stats["red_zone_rushing_tds_allowed"]
)


#use previously calculated Offense/Defense stats to determine 3rd Down Efficiency
offense_stats["third_down_attempts"] = (
    offense_stats["third_down_conversions"]
    + offense_stats["third_down_failures"]
)

offense_stats["third_down_conversion_rate"] = (
    offense_stats["third_down_conversions"]
    / offense_stats["third_down_attempts"].replace(0, float("nan"))
)

defense_stats["third_down_attempts_faced"] = (
    defense_stats["third_down_conversions_allowed"]
    + defense_stats["third_down_stops"]
)

defense_stats["third_down_conversion_rate_allowed"] = (
    defense_stats["third_down_conversions_allowed"]
    / defense_stats["third_down_attempts_faced"].replace(0, float("nan"))
)

#Get a Snapshot of the state of the game at the end of the 3rd Quarter
q3_situation = (
    df[
        df["qtr"].eq(4)
        & df["play_type"].notna()
    ]
    .sort_values(["season", "game_id", "play_id"])
    .groupby(["season", "game_id"])
    .head(1)
    [[
        "season", "game_id",
        "home_team", "away_team",
        "play_type", "quarter_seconds_remaining",
        "posteam", "down", "ydstogo", "yardline_100",
        "home_timeouts_remaining", "away_timeouts_remaining",
    ]]
    .reset_index(drop=True)
)

situation_cols = [
    "posteam", "down", "ydstogo", "yardline_100",
    "home_timeouts_remaining", "away_timeouts_remaining",
]
#Nulling values if the quarter_seconds_remaining is not equal to 900 (i.e. not the first play of the 4th quarter)
q3_situation.loc[
    q3_situation["quarter_seconds_remaining"].ne(900),
    situation_cols,
] = float("nan")

print(q3_situation.head(10).to_string(index=False))


#Calculate Team based Stats to determine stength of team - winning percantage, yards allowed, yards gained, etc.
team_games = (
    df.groupby(["season", "game_id", "posteam"])
    .agg(
        game_date=("game_date", "first"),
        home_team=("home_team", "first"),
        home_score=("home_score", "first"),
        away_score=("away_score", "first"),
        passing_yards=("passing_yards", "sum"),
        rushing_yards=("rushing_yards", "sum"),
    )
    .reset_index()
    .rename(columns={"posteam": "team"})
)

is_home = team_games["team"].eq(team_games["home_team"])

team_games["points_for"] = team_games["home_score"].where(
    is_home, team_games["away_score"]
)

team_games["points_against"] = team_games["away_score"].where(
    is_home, team_games["home_score"]
)

team_games["scoring_margin"] = (
    team_games["points_for"] - team_games["points_against"]
)

# A tie counts as half a win
team_games["win_credit"] = (
    (team_games["scoring_margin"] > 0).astype(float)
    + 0.5 * (team_games["scoring_margin"] == 0)
)

#Take Team states - apply it only to games that are already completed.
team_games = team_games.sort_values(
    ["season", "team", "game_date", "game_id"]
)

groups = team_games.groupby(["season", "team"])

team_games["pregame_games_played"] = groups.cumcount()

metrics = {
    "win_credit": "pregame_win_pct",
    "passing_yards": "pregame_passing_yards_per_game",
    "rushing_yards": "pregame_rushing_yards_per_game",
    "points_for": "pregame_points_per_game",
    "points_against": "pregame_points_allowed_per_game",
    "scoring_margin": "pregame_avg_scoring_margin",
}

for source, output in metrics.items():
    team_games[output] = groups[source].transform(
        lambda x: x.shift(1).expanding().mean()
    )



#Build Penalty Data Set
penalty_stats = (
    first_three_quarters[
        first_three_quarters["penalty"].eq(1)
    ]
    .groupby(["season", "game_id", "penalty_team", "qtr"])
    .size()
    .reset_index(name="penalty_plays")
    .rename(columns={"penalty_team": "team"})
)


#Assemble all the Stats into a DF - a row represents one team in one game
score_frames = []

for side, other in [("home", "away"), ("away", "home")]:
    scores = quarter_scores[[
        "season", "game_id", "qtr",
        f"{side}_team",
        f"{side}_score_quarter_end",
        f"{other}_score_quarter_end",
        f"{side}_points_scored_in_quarter",
        f"{side}_team_point_diff",
        f"{side}_team_qtr_point_change",
        "combined_score_quarter_end",
        "absolute_score_margin",
    ]].copy()

    scores.columns = [
        "season", "game_id", "qtr", "team",
        "points_for", "points_against", "points_scored",
        "point_diff", "point_diff_change",
        "combined_score", "absolute_margin",
    ]

    score_frames.append(scores)

quarterly = pd.concat(score_frames, ignore_index=True)


quarter_keys = ["season", "game_id", "team", "qtr"]

tables = [
    offense_stats,
    defense_stats,
    fumbles_lost_stats,
    return_td_stats,
    red_zone_offense_stats,
    red_zone_defense_stats,
    red_zone_td_stats,
    penalty_stats,
]

# Include this if you kept the separately renamed touchdown table
if "red_zone_defense_td_stats" in globals():
    tables.append(red_zone_defense_td_stats)

for table in tables:
    quarterly = quarterly.merge(
        table,
        on=quarter_keys,
        how="left",
        validate="one_to_one",
    )

# Missing event counts become zero; undefined rates remain missing
count_cols = [
    col
    for table in tables
    for col in table.columns
    if col not in quarter_keys and "rate" not in col
]

#Join team-quarter tables, turn Quarters into columns
quarterly[count_cols] = quarterly[count_cols].fillna(0)

model_data = quarterly.pivot(
    index=["season", "game_id", "team"],
    columns="qtr",
)

model_data.columns = [
    f"{stat}_q{int(quarter)}"
    for stat, quarter in model_data.columns
]

model_data = model_data.reset_index()

#Add Pregames Strenght of Team Stats
team_keys = ["season", "game_id", "team"]

pregame_cols = [
    col for col in team_games.columns
    if col.startswith("pregame_")
]

model_data = model_data.merge(
    team_games[team_keys + pregame_cols],
    on=team_keys,
    how="left",
    validate="one_to_one",
)

snapshot = (
    q3_situation
    .drop(columns=["play_type", "quarter_seconds_remaining"])
    .rename(columns={
        "posteam": "possession_team_q3",
        "down": "possession_down_q3",
        "ydstogo": "possession_yards_to_go_q3",
        "yardline_100": "possession_yards_to_goal_q3",
        "home_timeouts_remaining": "home_timeouts_q3",
        "away_timeouts_remaining": "away_timeouts_q3",
    })
)
model_data = model_data.merge(
    snapshot,
    on=["season", "game_id"],
    how="left",
    validate="many_to_one",
)

model_data["is_home"] = model_data["team"].eq(model_data["home_team"])

model_data["has_possession_q3"] = (
    model_data["team"].eq(model_data["possession_team_q3"])
    .astype(float)
    .where(model_data["possession_team_q3"].notna())
)

labels = team_games[team_keys + ["scoring_margin"]].copy()

labels["won"] = (
    labels["scoring_margin"].gt(0).astype(float)
    .where(
        labels["scoring_margin"].notna()
        & labels["scoring_margin"].ne(0)
    )
)

model_data = model_data.merge(
    labels[team_keys + ["won"]],
    on=team_keys,
    how="left",
    validate="one_to_one",
)

print(model_data.shape)
print(model_data.head().to_string(index=False))

#print (quarter_scores.head(20).to_string(index=False))

 season         game_id home_team away_team play_type  quarter_seconds_remaining posteam  down  ydstogo  yardline_100  home_timeouts_remaining  away_timeouts_remaining
   2019 2019_01_ATL_MIN       MIN       ATL       run                      900.0     MIN   1.0     10.0          75.0                      2.0                      2.0
   2019 2019_01_BAL_MIA       MIA       BAL      pass                      900.0     MIA   2.0      5.0          70.0                      3.0                      3.0
   2019 2019_01_BUF_NYJ       NYJ       BUF      punt                      900.0     NYJ   4.0      3.0          56.0                      2.0                      3.0
   2019 2019_01_CIN_SEA       SEA       CIN      pass                      900.0     SEA   1.0     20.0          44.0                      3.0                      3.0
   2019 2019_01_DEN_OAK        LV       DEN      punt                      900.0     DEN   4.0      3.0          88.0                      2.0                  